# In-Class Coding — Gaussian · KDE · GMM
### Lec4 · Spring 2026

**Instructions**
- Work through each problem in order; later problems build on earlier ones.
- Every cell marked `# ── YOUR CODE HERE ──` requires you to fill in the missing logic.
- Do **not** import `sklearn` for the core problems — implement everything from scratch.
- Run the **test cell** after each implementation to check your work.

---
**Three sections — three classes you will build step by step:**

| Section | Model | Class you build |
|---------|-------|-----------------|
| 1 | Single Gaussian | `Gaussian` |
| 2 | Kernel Density Estimation | `KDE` |
| 3 | Gaussian Mixture Model | `GMM` |

Each problem implements **one standalone function** (+ a test cell).
The **Assembly** cell at the end of each section wires your functions into an OOP class.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm as sp_norm

np.random.seed(42)
print("Ready.")

---
# Section 1 — The Gaussian Distribution

## Background

A 1-D Gaussian (Normal) distribution is fully described by two parameters:

$$\mathcal{N}(x \mid \mu, \sigma) = \frac{1}{\sigma\sqrt{2\pi}}
\exp\!\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

- $\mu$ — the **mean** (centre of the bell curve)
- $\sigma$ — the **standard deviation** (width of the bell curve)

In this section you will implement the three core operations of a Gaussian:

| Problem | Function | What it computes |
|---------|----------|-----------------|
| 1.1 | `gaussian_pdf` | Density at a point |
| 1.2 | `gaussian_log_likelihood` | How well parameters explain data |
| 1.3 | `gaussian_mle` | Best parameters given data |

## Problem 1.1 — Implement the Gaussian PDF

Implement the formula directly — **do not use** `scipy.stats.norm`.

$$\mathcal{N}(x \mid \mu, \sigma) = \frac{1}{\sigma\sqrt{2\pi}}
\exp\!\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

`x`, `mu`, `sigma` may be scalars or NumPy arrays; your function must handle both.

In [ ]:
def gaussian_pdf(x, mu, sigma):
    '''
    Evaluate the Gaussian PDF.

    Parameters
    ----------
    x     : float or np.ndarray
    mu    : float  — mean
    sigma : float  — standard deviation (> 0)

    Returns
    -------
    float or np.ndarray — same shape as x
    '''
    # ── YOUR CODE HERE ──────────────────────────────────────────────────


    # ────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test 1.1 ──────────────────────────────────────────────────────────────────
x_grid = np.linspace(-5, 5, 300)

assert np.allclose(gaussian_pdf(x_grid, 0, 1),
                   sp_norm.pdf(x_grid, 0, 1), atol=1e-10), \
    "1.1 FAIL: mismatch with scipy for N(0,1)"
assert np.allclose(gaussian_pdf(x_grid, -1, 2),
                   sp_norm.pdf(x_grid, -1, 2), atol=1e-10), \
    "1.1 FAIL: mismatch with scipy for N(-1,2)"
assert abs(gaussian_pdf(0.0, 0.0, 1.0) - 1/np.sqrt(2*np.pi)) < 1e-10, \
    "1.1 FAIL: N(0|0,1) should equal 1/sqrt(2pi)"

print("Test 1.1 PASSED ✓")

## Problem 1.2 — Log-Likelihood of Data

Given a dataset $\mathbf{x} = (x_1, \ldots, x_N)$ and candidate parameters
$(\mu, \sigma)$, the **log-likelihood** measures how well the Gaussian explains
the data:

$$\ell(\mu, \sigma) = \sum_{i=1}^{N} \log \mathcal{N}(x_i \mid \mu, \sigma)$$

A higher log-likelihood means the candidate parameters are a better fit.

**Your task:** implement this function, then use the provided visualisation
to see how the log-likelihood changes as $\mu$ varies.

In [ ]:
def gaussian_log_likelihood(x_data, mu, sigma):
    '''
    Compute the total log-likelihood of x_data under N(mu, sigma).

    Parameters
    ----------
    x_data : np.ndarray, shape (N,)
    mu     : float
    sigma  : float

    Returns
    -------
    float — sum of log-densities
    '''
    # ── YOUR CODE HERE ──────────────────────────────────────────────────


    # ────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test 1.2 ──────────────────────────────────────────────────────────────────
np.random.seed(7)
x_true_mu, x_true_sigma = 2.0, 1.5
data_s1 = np.random.normal(x_true_mu, x_true_sigma, 200)

ll_true = gaussian_log_likelihood(data_s1, x_true_mu, x_true_sigma)
ll_off  = gaussian_log_likelihood(data_s1, 0.0, 1.0)
assert np.isfinite(ll_true), "1.2 FAIL: log-likelihood should be finite"
assert ll_true > ll_off, "1.2 FAIL: true params should have higher LL than (0,1)"

print(f"LL at true params  (μ=2.0, σ=1.5) : {ll_true:.2f}")
print(f"LL at wrong params (μ=0.0, σ=1.0) : {ll_off:.2f}")
print("Test 1.2 PASSED ✓")

In [ ]:
# ── Visualise: LL as a function of μ (read-only, nothing to implement) ────────
mu_grid = np.linspace(-1, 5, 200)
ll_curve = [gaussian_log_likelihood(data_s1, m, x_true_sigma) for m in mu_grid]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: candidate Gaussians on data
ax = axes[0]
ax.hist(data_s1, bins=30, density=True, alpha=0.4, color="steelblue", label="Data")
x_plt = np.linspace(-4, 8, 400)
for mu_cand, col, lbl in [(-0.5, "red", "μ=−0.5 (bad)"),
                           (2.0,  "green", "μ=2.0 (true)"),
                           (3.5,  "orange", "μ=3.5 (off)")]:
    ll_c = gaussian_log_likelihood(data_s1, mu_cand, x_true_sigma)
    ax.plot(x_plt, gaussian_pdf(x_plt, mu_cand, x_true_sigma),
            color=col, lw=2, label=f"{lbl}  ℓ={ll_c:.0f}")
ax.set_title("Candidate Gaussians vs data")
ax.legend(fontsize=8)

# Right: LL curve
ax = axes[1]
ax.plot(mu_grid, ll_curve, "k-", lw=2)
ax.axvline(x_true_mu, color="green", ls="--", label=f"True μ = {x_true_mu}")
ax.axvline(mu_grid[np.argmax(ll_curve)], color="red", ls=":", label="LL peak")
ax.set_xlabel("μ  (σ fixed at 1.5)")
ax.set_ylabel("Log-likelihood")
ax.set_title("Log-likelihood landscape")
ax.legend()

plt.tight_layout()
plt.show()

**Answer the following (edit this cell):**

1.2a. Where does the log-likelihood curve peak relative to the true $\mu$?
   > *Your answer:*

1.2b. What happens to the log-likelihood when $\mu$ is very far from the data mean?
   > *Your answer:*

## Problem 1.3 — Maximum Likelihood Estimation (MLE)

In Lec2 we derived the closed-form MLE for a Gaussian:

$$\hat{\mu} = \frac{1}{N}\sum_{i=1}^{N} x_i, \qquad
\hat{\sigma} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(x_i - \hat{\mu})^2}$$

These are exactly the **sample mean** and **population standard deviation**.
Implement this below. Your function should return `(mu_hat, sigma_hat)`.

In [ ]:
def gaussian_mle(x_data):
    '''
    Fit a Gaussian to x_data by maximum likelihood estimation.

    Parameters
    ----------
    x_data : np.ndarray, shape (N,)

    Returns
    -------
    mu_hat    : float — MLE mean
    sigma_hat : float — MLE standard deviation
    '''
    # ── YOUR CODE HERE ──────────────────────────────────────────────────


    # ────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test 1.3 ──────────────────────────────────────────────────────────────────
mu_hat, sigma_hat = gaussian_mle(data_s1)

assert abs(mu_hat    - x_true_mu)    < 0.3, \
    f"1.3 FAIL: mu_hat={mu_hat:.3f}, expected near {x_true_mu}"
assert abs(sigma_hat - x_true_sigma) < 0.3, \
    f"1.3 FAIL: sigma_hat={sigma_hat:.3f}, expected near {x_true_sigma}"

ll_mle = gaussian_log_likelihood(data_s1, mu_hat, sigma_hat)
ll_off  = gaussian_log_likelihood(data_s1, x_true_mu + 1, x_true_sigma)
assert ll_mle >= ll_off, "1.3 FAIL: MLE params should have at least as high LL as a perturbed point"

print(f"MLE:  μ̂ = {mu_hat:.4f}  (true {x_true_mu})")
print(f"      σ̂ = {sigma_hat:.4f}  (true {x_true_sigma})")
print(f"LL at MLE: {ll_mle:.2f}")
print("Test 1.3 PASSED ✓")

## Assembly 1 — The `Gaussian` Class

Your three functions (`gaussian_pdf`, `gaussian_log_likelihood`, `gaussian_mle`)
are now the building blocks of a `Gaussian` class.
The cell below wires them together — **no new code to write**.

Read the class carefully: notice how each method calls exactly the function you
implemented, and how `fit` updates the object's own `mu` and `sigma`.

In [ ]:
class Gaussian:
    '''A 1-D Gaussian (Normal) distribution.'''

    def __init__(self, mu=0.0, sigma=1.0):
        self.mu    = float(mu)
        self.sigma = float(sigma)

    # ── Problem 1.1 ──────────────────────────────────────────────────────
    def pdf(self, x):
        '''Evaluate the Gaussian PDF at x (scalar or array).'''
        return gaussian_pdf(x, self.mu, self.sigma)

    # ── Problem 1.2 ──────────────────────────────────────────────────────
    def log_likelihood(self, x_data):
        '''Total log-likelihood of x_data under this Gaussian.'''
        return gaussian_log_likelihood(x_data, self.mu, self.sigma)

    # ── Problem 1.3 ──────────────────────────────────────────────────────
    def fit(self, x_data):
        '''Fit μ and σ to x_data via MLE. Updates self in-place, returns self.'''
        self.mu, self.sigma = gaussian_mle(x_data)
        return self

    # ── Provided ─────────────────────────────────────────────────────────
    def sample(self, n, seed=None):
        '''Draw n random samples from N(mu, sigma).'''
        rng = np.random.default_rng(seed)
        return rng.normal(self.mu, self.sigma, n)

    def __repr__(self):
        return f"Gaussian(μ={self.mu:.4f}, σ={self.sigma:.4f})"

In [ ]:
# ── Assembly 1 test ───────────────────────────────────────────────────────────
g = Gaussian().fit(data_s1)
print(g)

x_plt = np.linspace(-4, 8, 400)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(data_s1, bins=30, density=True, alpha=0.4, color="steelblue", label="Data")
ax.plot(x_plt, g.pdf(x_plt), "k-", lw=2, label=f"Fitted {g}")
ax.set_title("Gaussian.fit() — MLE result")
ax.legend()
plt.tight_layout()
plt.show()

assert abs(g.mu - x_true_mu) < 0.3 and abs(g.sigma - x_true_sigma) < 0.3, \
    "Assembly 1 FAIL: fitted params not close to true values"
print("Assembly 1 PASSED ✓")

---
# Section 2 — Kernel Density Estimation

## Background

A single Gaussian assumes the data has **one** bell-shaped mode.
When data has multiple clusters, this is too restrictive.

**Kernel Density Estimation (KDE)** solves this by placing one Gaussian
(a "kernel") at *every* data point and averaging them:

$$\hat{p}(x) = \frac{1}{N} \sum_{i=1}^{N} \mathcal{N}(x \mid x_i,\, h)$$

The single parameter $h > 0$ is the **bandwidth** — the width of each kernel.

| Problem | Function | What it computes |
|---------|----------|-----------------|
| 2.1 | `kde_pdf` | Density at query points |
| 2.2 | `kde_log_likelihood` | How well bandwidth $h$ fits held-out data |
| 2.3 | `kde_loo_bandwidth` | Best $h$ via leave-one-out cross-validation |

## Problem 2.1 — Implement the KDE Density

Given $N$ data points and bandwidth $h$, evaluate the KDE density at
each point in `x_eval`:

$$\hat{p}(x) = \frac{1}{N} \sum_{i=1}^{N} \mathcal{N}(x \mid x_i,\, h)$$

Use your `gaussian_pdf` from Problem 1.1.
`x_eval` is an array of $M$ query points; return an array of shape $(M,)$.

**Hint:** a vectorised approach uses broadcasting — think of shape $(M, N)$.

In [ ]:
def kde_pdf(x_eval, x_data, h):
    '''
    Evaluate a Gaussian KDE at query points x_eval.

    Parameters
    ----------
    x_eval : np.ndarray, shape (M,) — query points
    x_data : np.ndarray, shape (N,) — training data
    h      : float                  — bandwidth (> 0)

    Returns
    -------
    np.ndarray, shape (M,) — estimated density at each query point
    '''
    # ── YOUR CODE HERE ──────────────────────────────────────────────────
    # Hint: x_eval[:, None] has shape (M, 1); x_data[None, :] has shape (1, N).
    # gaussian_pdf applied to these broadcasts to (M, N) — average over axis=1.


    # ────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test 2.1 ──────────────────────────────────────────────────────────────────
from scipy.stats import gaussian_kde as sp_kde

np.random.seed(0)
data_s2 = np.concatenate([np.random.normal(-2, 0.8, 150),
                           np.random.normal( 2, 0.8, 150)])
x_eval  = np.linspace(-6, 6, 400)
h_test  = 0.5

your_density = kde_pdf(x_eval, data_s2, h_test)

# Must be non-negative
assert (your_density >= 0).all(), "2.1 FAIL: densities must be non-negative"

# Must integrate to ~1
area = np.trapz(your_density, x_eval)
assert 0.95 < area < 1.05, f"2.1 FAIL: integral = {area:.4f}, expected ~1"

# Must be close to scipy (which uses a slightly different normalisation convention
# but with bw_method='silverman' the shape should match your h=0.5 qualitatively)
sp_density = sp_kde(data_s2, bw_method=h_test / data_s2.std())(x_eval)
# Shape comparison: both should peak near x = ±2
assert np.argmax(your_density[:200]) > 50, \
    "2.1 FAIL: first peak should be near x=-2"
assert np.argmax(your_density[200:]) > 50, \
    "2.1 FAIL: second peak should be near x=+2"

print(f"Integral: {area:.4f}  (should be ~1)")
print("Test 2.1 PASSED ✓")

# Quick plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(data_s2, bins=30, density=True, alpha=0.35, color="steelblue", label="Data")
ax.plot(x_eval, your_density, "k-", lw=2, label=f"KDE  h={h_test}")
ax.set_title("KDE density (Problem 2.1)")
ax.legend()
plt.tight_layout()
plt.show()

## Problem 2.2 — KDE Log-Likelihood

To evaluate how well a bandwidth $h$ fits *unseen* data, compute the
**log-likelihood of test points** under the KDE fitted on training points:

$$\ell_{\text{test}}(h) = \sum_{x \in \mathcal{D}_{\text{test}}} \log \hat{p}_h(x)$$

where $\hat{p}_h$ is the KDE built from $\mathcal{D}_{\text{train}}$.

**Watch out:** if any density value is zero or near zero, $\log(0) = -\infty$.
Guard against this by adding a tiny $\epsilon = 10^{-300}$ before taking the log.

In [ ]:
def kde_log_likelihood(x_test, x_data, h):
    '''
    Log-likelihood of test points under a Gaussian KDE fitted on x_data.

    Parameters
    ----------
    x_test : np.ndarray, shape (M,) — held-out test points
    x_data : np.ndarray, shape (N,) — training data used to build the KDE
    h      : float                  — bandwidth

    Returns
    -------
    float — total log-likelihood (sum over test points)
    '''
    # ── YOUR CODE HERE ──────────────────────────────────────────────────
    # Step 1: evaluate kde_pdf at x_test (using x_data as training set)
    # Step 2: take log (guard: add 1e-300 before log)
    # Step 3: sum and return


    # ────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test 2.2 ──────────────────────────────────────────────────────────────────
np.random.seed(1)
idx     = np.random.permutation(len(data_s2))
n_tr    = int(0.8 * len(data_s2))
x_tr, x_te = data_s2[idx[:n_tr]], data_s2[idx[n_tr:]]

ll_good = kde_log_likelihood(x_te, x_tr, h=0.5)   # reasonable h
ll_bad  = kde_log_likelihood(x_te, x_tr, h=10.0)  # way too wide

assert np.isfinite(ll_good), "2.2 FAIL: log-likelihood should be finite"
assert np.isfinite(ll_bad),  "2.2 FAIL: log-likelihood should be finite"
assert ll_good > ll_bad, \
    f"2.2 FAIL: h=0.5 should beat h=10 on this data (got {ll_good:.1f} vs {ll_bad:.1f})"

print(f"LL (h=0.5) : {ll_good:.2f}")
print(f"LL (h=10)  : {ll_bad:.2f}")
print("Test 2.2 PASSED ✓")

## Problem 2.3 — Leave-One-Out Bandwidth Selection

A train/test split wastes data. **Leave-one-out (LOO)** cross-validation uses
all $N$ points for both fitting and evaluation: for each point $x_i$, build a
KDE from the remaining $N-1$ points and score $x_i$ on it.

$$\ell_{\text{LOO}}(h) = \sum_{i=1}^{N}
\log \hat{p}_{-i}(x_i), \qquad
\hat{p}_{-i}(x_i) = \frac{1}{N-1}\sum_{j \neq i} \mathcal{N}(x_i \mid x_j, h)$$

**Efficient trick:** the full KDE includes the $j = i$ self-term
$\mathcal{N}(x_i \mid x_i, h) = \frac{1}{h\sqrt{2\pi}}$ (same for all $i$).
You can compute the full $(N \times N)$ kernel matrix once and subtract the
diagonal to get the LOO estimates without an inner loop.

Implement `kde_loo_bandwidth`: given data and a grid of candidate $h$ values,
return the $h$ with the highest $\ell_{\text{LOO}}$.

In [ ]:
def kde_loo_bandwidth(x_data, h_values):
    '''
    Select bandwidth h by leave-one-out cross-validation.

    Parameters
    ----------
    x_data   : np.ndarray, shape (N,)
    h_values : np.ndarray          — candidate bandwidth values to search over

    Returns
    -------
    h_best    : float              — bandwidth with highest LOO log-likelihood
    loo_lls   : np.ndarray         — LOO log-likelihood for each h in h_values
    '''
    N = len(x_data)
    loo_lls = np.zeros(len(h_values))

    for idx_h, h in enumerate(h_values):
        # ── YOUR CODE HERE ───────────────────────────────────────────────
        # Option A (simple): loop over i; for each i, build KDE from x_data
        #   excluding index i, evaluate at x_data[i], accumulate log.
        #
        # Option B (fast / vectorised):
        #   1. Build kernel matrix K[i,j] = gaussian_pdf(x_data[i], x_data[j], h)
        #      shape (N, N).  Hint: use broadcasting.
        #   2. Sum each row and subtract the diagonal (self-term = 1/(h*sqrt(2pi))).
        #   3. Divide by (N-1) to get p_{-i}(x_i) for each i.
        #   4. Take log (guard against log(0)), sum.


        # ─────────────────────────────────────────────────────────────────

    h_best = h_values[np.argmax(loo_lls)]
    return h_best, loo_lls

In [ ]:
# ── Test 2.3 ──────────────────────────────────────────────────────────────────
h_grid = np.logspace(-1, 1, 40)
h_best, loo_lls = kde_loo_bandwidth(data_s2, h_grid)

assert np.isfinite(loo_lls).all(), \
    "2.3 FAIL: some LOO log-likelihoods are not finite"
assert 0.2 < h_best < 2.0, \
    f"2.3 FAIL: best h = {h_best:.3f}, expected in (0.2, 2.0)"

# Silverman's rule for comparison
h_sil = 1.06 * data_s2.std() * len(data_s2)**(-0.2)

print(f"LOO-optimal h    : {h_best:.4f}")
print(f"Silverman's rule : {h_sil:.4f}")
print("Test 2.3 PASSED ✓")

# Plot LOO curve
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.semilogx(h_grid, loo_lls, "k-", lw=2)
ax.axvline(h_best, color="green",  ls="--", lw=2, label=f"LOO best h = {h_best:.3f}")
ax.axvline(h_sil,  color="orange", ls=":",  lw=2, label=f"Silverman  h = {h_sil:.3f}")
ax.set_xlabel("Bandwidth h  (log scale)")
ax.set_ylabel("LOO log-likelihood")
ax.set_title("Bandwidth selection via LOO")
ax.legend()

ax = axes[1]
ax.hist(data_s2, bins=30, density=True, alpha=0.35, color="steelblue", label="Data")
x_plt = np.linspace(-6, 6, 400)
ax.plot(x_plt, kde_pdf(x_plt, data_s2, h_best), "g-", lw=2,
        label=f"KDE  h={h_best:.3f} (LOO)")
ax.plot(x_plt, kde_pdf(x_plt, data_s2, h_sil),  "orange", lw=2, ls="--",
        label=f"KDE  h={h_sil:.3f} (Silverman)")
ax.set_title("KDE with selected bandwidth")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Answer the following (edit this cell):**

2.3a. Is the LOO-optimal $h$ close to Silverman's rule estimate? Which is larger?
   > *Your answer:*

2.3b. What does the KDE look like when $h$ is very small vs. very large?
      Which statistical terms (bias, variance) describe these two extremes?
   > *Your answer:*

2.3c. Why is leave-one-out preferable to a simple 80/20 train/test split
      for bandwidth selection?
   > *Your answer:*

## Assembly 2 — The `KDE` Class

Your three KDE functions are now packaged as a class with a clean interface.
**No new code to write** — read and run the cell.

In [ ]:
class KDE:
    '''Gaussian Kernel Density Estimator.'''

    def __init__(self, h=1.0):
        self.h       = float(h)
        self._x_data = None

    # ── Provided — data storage ──────────────────────────────────────────
    def fit(self, x_data):
        '''Store training data (bandwidth unchanged).'''
        self._x_data = np.asarray(x_data, dtype=float)
        return self

    # ── Problem 2.1 ──────────────────────────────────────────────────────
    def pdf(self, x_eval):
        '''Evaluate KDE density at query points x_eval.'''
        return kde_pdf(np.asarray(x_eval), self._x_data, self.h)

    # ── Problem 2.2 ──────────────────────────────────────────────────────
    def log_likelihood(self, x_test):
        '''Log-likelihood of x_test under this KDE.'''
        return kde_log_likelihood(np.asarray(x_test), self._x_data, self.h)

    # ── Problem 2.3 ──────────────────────────────────────────────────────
    def fit_bandwidth(self, x_data, h_values=None):
        '''Select h by LOO cross-validation, then store training data.'''
        x_data = np.asarray(x_data, dtype=float)
        if h_values is None:
            h_values = np.logspace(-1, 1, 40)
        self.h, _ = kde_loo_bandwidth(x_data, h_values)
        self._x_data = x_data
        return self

    # ── Provided — sampling ──────────────────────────────────────────────
    def sample(self, n, seed=None):
        '''Draw n samples: pick a training point, add Gaussian noise of width h.'''
        rng = np.random.default_rng(seed)
        centres = rng.choice(self._x_data, size=n, replace=True)
        return centres + self.h * rng.standard_normal(n)

    def __repr__(self):
        n = len(self._x_data) if self._x_data is not None else 0
        return f"KDE(h={self.h:.4f}, N={n})"

In [ ]:
# ── Assembly 2 test ───────────────────────────────────────────────────────────
kde_model = KDE().fit_bandwidth(data_s2)
print(kde_model)

x_plt = np.linspace(-6, 6, 400)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(data_s2, bins=30, density=True, alpha=0.35, color="steelblue", label="Data")
ax.plot(x_plt, kde_model.pdf(x_plt), "k-", lw=2, label=f"Fitted {kde_model}")
ax.set_title("KDE.fit_bandwidth() — LOO result")
ax.legend()
plt.tight_layout()
plt.show()

assert 0.2 < kde_model.h < 2.0, "Assembly 2 FAIL: selected h out of range"
print("Assembly 2 PASSED ✓")

---
# Section 3 — Gaussian Mixture Models

## Background

Both a single Gaussian and KDE are limiting cases:
- **Single Gaussian:** 1 component, parameters fit by closed-form MLE.
- **KDE:** $N$ components (one per data point), bandwidth selected by LOO.

A **Gaussian Mixture Model (GMM)** sits in between: $K$ components,
each with its own mean $\mu_k$, standard deviation $\sigma_k$, and
mixing weight $\pi_k$:

$$p(x) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(x \mid \mu_k, \sigma_k),
\qquad \sum_{k=1}^K \pi_k = 1, \quad \pi_k > 0$$

Parameters are found by **Expectation-Maximization (EM)**.

| Problem | Function | What it computes |
|---------|----------|-----------------|
| 3.1 | `gmm_sample` | Draw samples from a GMM |
| 3.2 | `gmm_e_step` | Responsibility of each component for each point |
| 3.3 | `gmm_m_step` | Update parameters given responsibilities |
| 3.4 | `gmm_log_likelihood` | GMM log-likelihood (numerically stable) |

In [ ]:
# ── Shared dataset for Section 3 ──────────────────────────────────────────────
np.random.seed(0)

pis_true    = np.array([0.3, 0.5, 0.2])
mus_true    = np.array([-4.0, 0.5, 4.0])
sigmas_true = np.array([0.8,  1.5, 0.6])
K_true, N   = 3, 500

z_true  = np.random.choice(K_true, size=N, p=pis_true)
x_data3 = np.array([np.random.normal(mus_true[k], sigmas_true[k])
                     for k in z_true])

print(f"Dataset: N={N} from a {K_true}-component GMM")
print(f"  π = {pis_true}  μ = {mus_true}  σ = {sigmas_true}")

## Problem 3.1 — Sample from a GMM

The GMM is a **generative model**: to draw one sample, follow two steps.

1. **Pick a component** $k$ by sampling from a categorical distribution:
   $k \sim \text{Categorical}(\boldsymbol{\pi})$
2. **Draw from that component:**
   $x \sim \mathcal{N}(\mu_k, \sigma_k)$

Repeat $n$ times.

**Hint:** `np.random.choice(K, size=n, p=pis)` handles Step 1.

In [ ]:
def gmm_sample(pis, mus, sigmas, n, seed=None):
    '''
    Draw n samples from a Gaussian Mixture Model.

    Parameters
    ----------
    pis    : np.ndarray, shape (K,) — mixing weights (sum to 1)
    mus    : np.ndarray, shape (K,) — component means
    sigmas : np.ndarray, shape (K,) — component standard deviations
    n      : int                    — number of samples
    seed   : int or None

    Returns
    -------
    samples     : np.ndarray, shape (n,) — generated values
    assignments : np.ndarray, shape (n,) — component index for each sample
    '''
    if seed is not None:
        np.random.seed(seed)
    K = len(pis)

    # ── YOUR CODE HERE ──────────────────────────────────────────────────
    # Step 1: draw n component indices (assignments) from Categorical(pis)
    # Step 2: for each i, sample x_i ~ N(mus[assignments[i]], sigmas[assignments[i]])


    # ────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test 3.1 ──────────────────────────────────────────────────────────────────
samples_g, assigns_g = gmm_sample(pis_true, mus_true, sigmas_true, n=2000, seed=99)

assert samples_g.shape  == (2000,), "3.1 FAIL: samples shape wrong"
assert assigns_g.shape  == (2000,), "3.1 FAIL: assignments shape wrong"
assert set(assigns_g).issubset({0, 1, 2}), "3.1 FAIL: assignments must be in {0,1,2}"

for k in range(K_true):
    frac = (assigns_g == k).mean()
    assert abs(frac - pis_true[k]) < 0.06, \
        f"3.1 FAIL: component {k} fraction {frac:.3f} far from π={pis_true[k]}"

print("Test 3.1 PASSED ✓")

# Visualise
colors_k = ["tab:blue", "tab:orange", "tab:green"]
x_plt = np.linspace(-8, 8, 500)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
for k in range(K_true):
    ax.hist(samples_g[assigns_g == k], bins=40, alpha=0.55,
            color=colors_k[k], label=f"k={k}  π={pis_true[k]}")
ax.set_title("GMM samples coloured by component")
ax.legend()

ax = axes[1]
ax.hist(samples_g, bins=60, density=True, alpha=0.4, color="gray", label="Samples")
true_den = sum(pis_true[k] * gaussian_pdf(x_plt, mus_true[k], sigmas_true[k])
               for k in range(K_true))
ax.plot(x_plt, true_den, "k-", lw=2, label="True GMM density")
ax.set_title("Samples vs. true density")
ax.legend()

plt.tight_layout()
plt.show()

**Answer the following (edit this cell):**

3.1a. Do the sample fractions per component match the true mixing weights $\boldsymbol{\pi}$?
   > *Your answer:*

3.1b. How does GMM sampling (pick a learned component, then sample) differ from
      KDE sampling (pick a training data point, add noise)?
   > *Your answer:*

## Problem 3.2 — E-step: Responsibilities

In the **E-step** we compute how much each component $k$ "claims" each
data point $x_i$. This is called the **responsibility**:

$$r_{ik} = \frac{\pi_k \, \mathcal{N}(x_i \mid \mu_k, \sigma_k)}
{\displaystyle\sum_{j=1}^{K} \pi_j \, \mathcal{N}(x_i \mid \mu_j, \sigma_j)}$$

- $r_{ik} \in [0, 1]$ for all $i, k$.
- $\sum_k r_{ik} = 1$ for every data point $i$ (**each row sums to 1**).
- $r_{ik} \approx 1$ means component $k$ almost certainly generated $x_i$.

Return a matrix $R$ of shape $(N, K)$ where $R[i, k] = r_{ik}$.

In [ ]:
def gmm_e_step(x, pis, mus, sigmas):
    '''
    E-step: compute responsibilities r[i, k] for each point and component.

    Parameters
    ----------
    x      : np.ndarray, shape (N,)
    pis    : np.ndarray, shape (K,) — mixing weights
    mus    : np.ndarray, shape (K,) — component means
    sigmas : np.ndarray, shape (K,) — component std devs

    Returns
    -------
    r : np.ndarray, shape (N, K) — each row sums to 1
    '''
    N, K = len(x), len(pis)

    # ── YOUR CODE HERE ──────────────────────────────────────────────────
    # Step 1: compute numerator[i, k] = pis[k] * gaussian_pdf(x[i], mus[k], sigmas[k])
    #         Hint: shape (N, K) via broadcasting.
    # Step 2: divide each row by its sum to normalise.


    # ────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test 3.2 ──────────────────────────────────────────────────────────────────
r_test = gmm_e_step(x_data3, pis_true, mus_true, sigmas_true)

assert r_test.shape == (N, K_true), \
    f"3.2 FAIL: shape should be ({N}, {K_true}), got {r_test.shape}"
assert (r_test >= 0).all(), \
    "3.2 FAIL: responsibilities must be non-negative"
assert np.allclose(r_test.sum(axis=1), 1.0, atol=1e-8), \
    f"3.2 FAIL: rows must sum to 1; max deviation = {abs(r_test.sum(axis=1)-1).max():.2e}"

# Sanity: a point near mu=-4 should be dominated by component 0
idx_near_k0 = np.argmin(np.abs(x_data3 - mus_true[0]))
assert r_test[idx_near_k0, 0] > 0.8, \
    f"3.2 FAIL: point near μ₀=-4 should have r[:,0]>0.8, got {r_test[idx_near_k0,0]:.3f}"

print(f"Responsibility shape : {r_test.shape}")
print(f"Row-sum range : [{r_test.sum(axis=1).min():.8f}, {r_test.sum(axis=1).max():.8f}]")
print("Test 3.2 PASSED ✓")

## Problem 3.3 — M-step: Parameter Updates

Given responsibilities $r_{ik}$, update each parameter using the
closed-form maximisers derived in Lec3:

$$N_k = \sum_{i=1}^{N} r_{ik}$$

$$\pi_k^{\text{new}} = \frac{N_k}{N}, \qquad
\mu_k^{\text{new}} = \frac{\sum_i r_{ik}\, x_i}{N_k}, \qquad
\sigma_k^{\text{new}} = \sqrt{\frac{\sum_i r_{ik}(x_i - \mu_k^{\text{new}})^2}{N_k}}$$

In [ ]:
def gmm_m_step(x, r):
    '''
    M-step: update GMM parameters given responsibilities.

    Parameters
    ----------
    x : np.ndarray, shape (N,)
    r : np.ndarray, shape (N, K) — responsibilities from E-step

    Returns
    -------
    new_pis    : np.ndarray, shape (K,)
    new_mus    : np.ndarray, shape (K,)
    new_sigmas : np.ndarray, shape (K,)
    '''
    N, K = r.shape

    # ── YOUR CODE HERE ──────────────────────────────────────────────────
    # Step 1: effective count  Nk[k] = r[:, k].sum()
    # Step 2: new_pis = Nk / N
    # Step 3: new_mus[k] = (r[:, k] * x).sum() / Nk[k]
    # Step 4: new_sigmas[k] = sqrt( (r[:, k] * (x - new_mus[k])**2).sum() / Nk[k] )


    # ────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test 3.3 ──────────────────────────────────────────────────────────────────
r_from_true = gmm_e_step(x_data3, pis_true, mus_true, sigmas_true)
new_pis, new_mus, new_sigmas = gmm_m_step(x_data3, r_from_true)

assert new_pis.shape    == (K_true,), "3.3 FAIL: pis shape wrong"
assert new_mus.shape    == (K_true,), "3.3 FAIL: mus shape wrong"
assert new_sigmas.shape == (K_true,), "3.3 FAIL: sigmas shape wrong"
assert np.allclose(new_pis.sum(), 1.0, atol=1e-8), \
    f"3.3 FAIL: new pis must sum to 1, got {new_pis.sum():.6f}"
assert (new_sigmas > 0).all(), "3.3 FAIL: sigmas must be positive"

# One M-step from true params should barely move them
for k in range(K_true):
    assert abs(new_mus[k] - mus_true[k]) < 0.3, \
        f"3.3 FAIL: mu[{k}] moved too far: {new_mus[k]:.3f} vs {mus_true[k]}"

print(f"π: {new_pis.round(3)}   (true: {pis_true})")
print(f"μ: {new_mus.round(3)}   (true: {mus_true})")
print(f"σ: {new_sigmas.round(3)} (true: {sigmas_true})")
print("Test 3.3 PASSED ✓")

## Problem 3.4 — GMM Log-Likelihood (with the Log-Sum-Exp Trick)

The GMM log-likelihood is:

$$\ell(\boldsymbol{\theta}) = \sum_{i=1}^{N}
\log \left[\sum_{k=1}^{K} \pi_k \, \mathcal{N}(x_i \mid \mu_k, \sigma_k)\right]$$

Computing $\log\!\sum_k e^{a_k}$ naively can underflow when $a_k$ is very negative.
Use the **log-sum-exp trick** for numerical stability:

$$\log\!\sum_k e^{a_k} = a_{\max} + \log\!\sum_k e^{a_k - a_{\max}},
\qquad a_{\max} = \max_k a_k$$

**Hint:** let $a_{ik} = \log \pi_k + \log \mathcal{N}(x_i \mid \mu_k, \sigma_k)$
and apply log-sum-exp over $k$ for each $i$.
Note that $\log \mathcal{N}(x \mid \mu, \sigma)
= -\log\sigma - \tfrac{1}{2}\log(2\pi) - \tfrac{1}{2}\!\left(\tfrac{x-\mu}{\sigma}\right)^2$.

In [ ]:
def gmm_log_likelihood(x, pis, mus, sigmas):
    '''
    Log-likelihood of data x under a GMM (numerically stable).

    Parameters
    ----------
    x      : np.ndarray, shape (N,)
    pis    : np.ndarray, shape (K,)
    mus    : np.ndarray, shape (K,)
    sigmas : np.ndarray, shape (K,)

    Returns
    -------
    float — total log-likelihood
    '''
    K = len(pis)

    # ── YOUR CODE HERE ──────────────────────────────────────────────────
    # Step 1: build log_terms[i, k] = log(pi_k) + log N(x_i | mu_k, sigma_k)
    #         shape (N, K).
    # Step 2: log-sum-exp over k (axis=1) for each i.
    # Step 3: sum over i.


    # ────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test 3.4 ──────────────────────────────────────────────────────────────────
ll_true_val = gmm_log_likelihood(x_data3, pis_true, mus_true, sigmas_true)
ll_bad_val  = gmm_log_likelihood(x_data3,
                                 np.array([1/3, 1/3, 1/3]),
                                 np.array([0., 0., 0.]),
                                 np.array([1., 1., 1.]))

assert np.isfinite(ll_true_val), "3.4 FAIL: LL should be finite"
assert ll_true_val > ll_bad_val, \
    f"3.4 FAIL: true params should beat flat init ({ll_true_val:.1f} vs {ll_bad_val:.1f})"

print(f"LL (true params)  : {ll_true_val:.2f}")
print(f"LL (flat init)    : {ll_bad_val:.2f}")
print("Test 3.4 PASSED ✓")

## Assembly 3 — The `GMM` Class

Your four functions are now assembled into a `GMM` class.
The `fit` method (provided) orchestrates the full EM loop — **no new code**.

Read the `fit` method carefully: it initialises parameters randomly, then
alternates E-step and M-step until the log-likelihood converges.

In [ ]:
class GMM:
    '''Gaussian Mixture Model fitted by Expectation-Maximization.'''

    def __init__(self, K):
        self.K          = K
        self.pis        = None
        self.mus        = None
        self.sigmas     = None
        self.ll_history = []

    # ── Problem 3.1 ──────────────────────────────────────────────────────
    def sample(self, n, seed=None):
        '''Draw n samples from the fitted GMM.'''
        return gmm_sample(self.pis, self.mus, self.sigmas, n, seed=seed)

    # ── Problem 3.2 ──────────────────────────────────────────────────────
    def _e_step(self, x):
        '''Compute responsibilities.'''
        return gmm_e_step(x, self.pis, self.mus, self.sigmas)

    # ── Problem 3.3 ──────────────────────────────────────────────────────
    def _m_step(self, x, r):
        '''Update parameters given responsibilities.'''
        self.pis, self.mus, self.sigmas = gmm_m_step(x, r)

    # ── Problem 3.4 ──────────────────────────────────────────────────────
    def log_likelihood(self, x):
        '''GMM log-likelihood of data x.'''
        return gmm_log_likelihood(x, self.pis, self.mus, self.sigmas)

    # ── Provided — EM loop ───────────────────────────────────────────────
    def fit(self, x, max_iter=200, tol=1e-6, seed=None):
        '''
        Run EM until convergence.

        Initialisation: equal weights, random means from data, shared std.
        Convergence: stop when |LL(t) - LL(t-1)| < tol.
        '''
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(x), self.K, replace=False)

        self.pis    = np.full(self.K, 1.0 / self.K)
        self.mus    = x[idx].copy()
        self.sigmas = np.full(self.K, x.std())

        self.ll_history = [self.log_likelihood(x)]

        for _ in range(max_iter):
            r  = self._e_step(x)
            self._m_step(x, r)
            ll = self.log_likelihood(x)
            self.ll_history.append(ll)
            if abs(ll - self.ll_history[-2]) < tol:
                break

        return self

    def __repr__(self):
        return f"GMM(K={self.K}, iters={len(self.ll_history)-1})"

In [ ]:
# ── Assembly 3 test ───────────────────────────────────────────────────────────
gmm_model = GMM(K=3).fit(x_data3, seed=7)
print(gmm_model)
print(f"  π: {gmm_model.pis.round(3)}   (true: {pis_true})")
print(f"  μ: {gmm_model.mus.round(3)}   (true: {mus_true})")
print(f"  σ: {gmm_model.sigmas.round(3)} (true: {sigmas_true})")

# LL must be non-decreasing
diffs = np.diff(gmm_model.ll_history)
assert (diffs >= -1e-6).all(), \
    f"Assembly 3 FAIL: LL decreased at iter(s) {np.where(diffs < -1e-6)[0]}"

# Final LL should be close to true-parameter LL
ll_true_val = gmm_log_likelihood(x_data3, pis_true, mus_true, sigmas_true)
assert gmm_model.ll_history[-1] >= ll_true_val - 10, \
    "Assembly 3 FAIL: final LL too far below true-params LL"

print(f"\nFinal LL: {gmm_model.ll_history[-1]:.2f}   (true-params LL: {ll_true_val:.2f})")
print("Assembly 3 PASSED ✓")

# Plot: LL convergence + fitted density
x_plt = np.linspace(-8, 8, 500)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(gmm_model.ll_history, "k-o", ms=3)
ax.set_xlabel("EM iteration")
ax.set_ylabel("Log-likelihood")
ax.set_title("EM convergence (must be non-decreasing)")

ax = axes[1]
ax.hist(x_data3, bins=40, density=True, alpha=0.35, color="steelblue", label="Data")
fitted_den = sum(gmm_model.pis[k] * gaussian_pdf(x_plt, gmm_model.mus[k],
                                                  gmm_model.sigmas[k])
                 for k in range(gmm_model.K))
ax.plot(x_plt, fitted_den, "k-", lw=2, label=f"Fitted {gmm_model}")
true_den   = sum(pis_true[k] * gaussian_pdf(x_plt, mus_true[k], sigmas_true[k])
                 for k in range(K_true))
ax.plot(x_plt, true_den, "g--", lw=1.5, label="True density")
ax.set_title("Fitted GMM vs true density")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Answer the following (edit this cell):**

3.4a. Does the EM log-likelihood curve look monotonically non-decreasing?
      Why is this a theoretical guarantee?
   > *Your answer:*

3.4b. ⭐ Run `GMM(K=3).fit(x_data3, seed=s)` for seeds `[0, 7, 13, 21, 42]`.
      Collect the final log-likelihood from each run and plot a bar chart
      (best run green, worst red).
      By how much does the final LL vary? What went wrong in the worst run?
   > *Your answer:*

---
# General Conceptual Questions

Answer all five questions below (edit this cell).

---

**Q1 — Comparing the Three Models**

You have a 1-D dataset with three well-separated clusters.
Rank the models — `Gaussian`, `KDE`, `GMM` — from worst to best fit,
and justify your ranking in 2–3 sentences.

> *Your answer:*

---

**Q2 — Likelihood as a Scoring Function**

In Section 1 (Problem 1.2) you plotted log-likelihood vs $\mu$.
In Section 2 (Problem 2.2–2.3) you used log-likelihood to select bandwidth $h$.
In Section 3 (Problem 3.4) you monitored log-likelihood during EM.

What single property makes log-likelihood useful in all three contexts?

> *Your answer:*

---

**Q3 — KDE vs GMM: Bias and Parameters**

(a) KDE has $N$ "components" (one per data point); GMM has $K \ll N$.
    Which model has more parameters, and what are the consequences for
    generalisation and memory?

(b) If you want to *interpret* the structure of a dataset (e.g. "there are
    3 distinct clusters with weights 0.3, 0.5, 0.2"), which model do you prefer?

> *Your answer:*

---

**Q4 — LOO vs MLE Bandwidth**

In Problem 2.3 you used leave-one-out cross-validation.
Explain why the naive approach — maximising $\ell(h) = \sum_i \log \hat{p}_h(x_i)$
*where $x_i$ is also in the KDE* — always prefers $h \to 0$.

> *Your answer:*

---

**Q5 — EM Local Optima**

EM is guaranteed to be non-decreasing in log-likelihood but is **not**
guaranteed to find the global maximum.

(a) What property of the EM objective causes local optima?
(b) Name one practical strategy (besides running multiple random restarts)
    to improve initialisation quality for GMM fitting.

> *Your answer:*